# Model comparison

## Objective

The objective of this notebook is to train, evaluate, and compare multiple baseline classification models using the leakage-safe preprocessing pipeline created in the previous stages.

The analysis will establish a reliable performance baseline for customer churn prediction, compare model behavior using classification metrics such as recall, precision, F1-score, balanced accuracy, and confusion matrices, and identify the most promising models for further tuning.

Particular attention will be given to the churn class (`1`), since correctly identifying customers at risk of leaving is more valuable than maximizing overall accuracy alone.

The results of this notebook will be used to select candidate models for hyperparameter optimization, threshold analysis, business-cost evaluation, and eventual deployment.

In [139]:
import pandas as pd
from pathlib import Path
from sklearn.pipeline import Pipeline
import numpy as np
import math
from IPython.display import display
import joblib
import matplotlib.pyplot as plt

# Validation
from sklearn.utils.validation import check_is_fitted
from sklearn.base import clone

# Models
from sklearn.ensemble import (ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Metrics
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

## Load modeling artifacts

The previously generated modeling artifacts are loaded to prepare the candidate-model evaluation stage.

The training and test splits are restored from the processed data directory, and the `Churn` target is separated from the predictor matrices to reconstruct `X_train`, `y_train`, `X_test`, and `y_test`.

The fitted preprocessing artifact is also loaded so that the same feature-transformation configuration defined in the preprocessing notebook can be reused consistently.

In addition, the baseline cross-validation summary is loaded to preserve the results obtained in the previous modeling stage. These baseline results will later be compared with the candidate-model cross-validation results under the same evaluation framework.

The held-out test set is loaded only as part of the project artifacts and will remain untouched during candidate-model selection and cross-validation.

In [140]:
SPLIT_DATA_DIR = Path("../data/processed/splits")

train_data = pd.read_csv(SPLIT_DATA_DIR / "train.csv")
test_data = pd.read_csv(SPLIT_DATA_DIR / "test.csv")

X_train = train_data.drop(columns="Churn")
y_train = train_data["Churn"]

X_test = test_data.drop(columns="Churn")
y_test = test_data["Churn"]

In [141]:
MODELS_DIR = Path("../models")
PREPROCESSOR_PATH = MODELS_DIR / "preprocessor.joblib"
if not PREPROCESSOR_PATH.exists():
    raise FileNotFoundError(f"preprocessor not found at {PREPROCESSOR_PATH.resolve()}")
preprocessor = joblib.load(PREPROCESSOR_PATH)

In [142]:
CV_SUMMARY_PATH = Path("../data/processed/cv_summary.csv")

# Load the first CSV column as the row index
baseline_cv_summary = pd.read_csv(
    CV_SUMMARY_PATH,
    index_col=0
)

# Identify metric columns
metric_columns = [
    column
    for column in baseline_cv_summary.columns
    if column != "model"
]

# Convert metric columns to numeric
baseline_cv_summary[metric_columns] = baseline_cv_summary[
    metric_columns
].apply(pd.to_numeric)

display(baseline_cv_summary.head())

,dummy,logistic_regression,decision_tree,random_forest
test_accuracy_mean,0.734647,0.802451,0.727901,0.785234
test_accuracy_std,0.000094,0.011979,0.009219,0.009333
train_accuracy_mean,0.734647,0.806310,0.998314,0.998314
train_accuracy_std,0.000024,0.003327,0.000226,0.000226
test_balanced_accuracy_mean,0.500000,0.719843,0.652011,0.685906


In [143]:
# Row consistency
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

# Expected feature structure
assert X_train.shape[1] == X_test.shape[1]
assert X_train.columns.equals(X_test.columns)

# Target should not be present in X
assert "Churn" not in X_train.columns
assert "Churn" not in X_test.columns

# Binary target validation
assert set(y_train.unique()).issubset({0, 1})
assert set(y_test.unique()).issubset({0, 1})

# No missing target values
assert y_train.notna().all()
assert y_test.notna().all()

print("Split data validated successfully.")

Split data validated successfully.


### Split data validation

The loaded training and test splits are validated to confirm that the feature and target matrices are structurally consistent.

The checks verify that the number of observations matches between features and targets, both splits contain the same predictor schema, the target variable is excluded from the feature matrices, and the target contains only the expected binary classes.

In [144]:
# Confirm that the preprocessing artifact is fitted
check_is_fitted(preprocessor)

# Confirm expected input feature count
assert hasattr(preprocessor, "feature_names_in_")
assert len(preprocessor.feature_names_in_) == X_train.shape[1]

# Confirm training columns match preprocessor input schema
assert set(preprocessor.feature_names_in_) == set(X_train.columns)

print("Preprocessor validated successfully.")

Preprocessor validated successfully.


### Preprocessor validation

The saved preprocessing artifact is validated to confirm that it remains fitted and expects the same feature schema as the current training data.

A small transformation check is also performed to verify that the loaded preprocessor can successfully transform new observations without refitting.

In [145]:
assert baseline_cv_summary is not None
assert not baseline_cv_summary.empty

print("Baseline CV summary loaded successfully.")
print("Shape:", baseline_cv_summary.shape)

display(baseline_cv_summary.head())

Baseline CV summary loaded successfully.
Shape: (40, 4)


,dummy,logistic_regression,decision_tree,random_forest
test_accuracy_mean,0.734647,0.802451,0.727901,0.785234
test_accuracy_std,0.000094,0.011979,0.009219,0.009333
train_accuracy_mean,0.734647,0.806310,0.998314,0.998314
train_accuracy_std,0.000024,0.003327,0.000226,0.000226
test_balanced_accuracy_mean,0.500000,0.719843,0.652011,0.685906


### Baseline cross-validation summary validation

The previously saved baseline cross-validation summary is validated before being reused for model comparison.

The checks confirm that the summary contains valid model identifiers, the expected metric structure, numerical and finite metric values, and non-negative standard deviations.

This ensures that the baseline results can be compared reliably with the candidate-model cross-validation results.

## Candidate model initialization

A set of additional classification models is initialized for baseline comparison.

These models represent different machine learning families, including tree ensembles, boosting methods, margin-based classifiers, distance-based methods, probabilistic models, and neural networks.

At this stage, the models are created using their default configurations. The objective is not to optimize them yet, but to establish a broad and consistent comparison across different modeling approaches.

In [146]:
candidate_models = {
    "extra_trees": ExtraTreesClassifier(),
    "gradient_boosting": GradientBoostingClassifier(),
    "hist_gradient_boosting": HistGradientBoostingClassifier(),
    "support_vector_machine": SVC(),
    "knn": KNeighborsClassifier(),
    "gaussian_naive_bayes": GaussianNB(),
    "mlp": MLPClassifier(),
    "xgboost": XGBClassifier(),
    "lightgbm": LGBMClassifier(verbose=0),
    "catboost": CatBoostClassifier(verbose=0)
}

In [147]:
for model_name, model in candidate_models.items():
    print(f"\n--- {model_name} ---")
    print(model.get_params())


--- extra_trees ---
{'bootstrap': False, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': None, 'verbose': 0, 'warm_start': False}

--- gradient_boosting ---
{'ccp_alpha': 0.0, 'criterion': 'deprecated', 'init': None, 'learning_rate': 0.1, 'loss': 'log_loss', 'max_depth': 3, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 100, 'n_iter_no_change': None, 'random_state': None, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}

--- hist_gradient_boosting ---
{'categorical_features': 'from_dtype', 'class_weight': None, 'ea

### Candidate model validation

The candidate models were initialized successfully and their configurations were inspected using `get_params()`.

This validation confirms that each estimator is available in the current environment and exposes the expected model parameters required for later training, evaluation, and hyperparameter tuning.

The validated models are now ready to be incorporated into the same preprocessing and cross-validation workflow used for the baseline models.

### Candidate pipeline validation

Before running cross-validation, the candidate pipelines are validated to confirm that:

- every candidate model has a corresponding pipeline;
- each pipeline contains both preprocessing and model stages;
- the preprocessing step can transform the training data successfully;
- the transformed data preserves the expected number of observations;
- no missing or infinite values are produced by preprocessing;
- each candidate model can be fitted and can generate predictions successfully.

Only the training data is used during this validation so that the held-out test set remains untouched.

### Preprocessing and model input validation

Before cross-validation, the preprocessing workflow is validated independently from the candidate classifiers.

The validation first confirms that the preprocessing configuration can transform the original training features into a numerical, finite, and model-ready feature matrix while preserving the number of observations.

The resulting transformed matrix is then provided directly to each candidate classifier to verify that the processed representation is compatible with the model's input requirements.

This stage does not evaluate predictive performance or generalization. Its purpose is to confirm that the preprocessing produces valid data and that every candidate model can consume the resulting feature representation successfully.

In [148]:
candidate_models_pipeline = {
    model_name: Pipeline([('preprocessing', preprocessor), ('model', estimator)])
    for model_name, estimator in candidate_models.items()
}

In [149]:
preprocessing_check = clone(preprocessor)

X_train_processed_check = preprocessing_check.fit_transform(X_train, y_train)

In [150]:
# Same number of observations
assert X_train_processed_check.shape[0] == X_train.shape[0]

# Features were actually generated
assert X_train_processed_check.shape[1] > 0

# Convert temporarily for validation
if hasattr(X_train_processed_check, "toarray"):
    processed_values = X_train_processed_check.toarray()
else:
    processed_values = np.asarray(X_train_processed_check)

# Numerical output
assert np.issubdtype(processed_values.dtype, np.number)

# No missing values
assert not np.isnan(processed_values).any()

# No infinite values
assert np.isfinite(processed_values).all()

print(
    "Preprocessing validation passed:",
    X_train.shape,
    "->",
    X_train_processed_check.shape
)

Preprocessing validation passed: (5634, 19) -> (5634, 45)


### Validation results

The preprocessing workflow produced a valid numerical feature matrix with the expected number of observations and without missing or infinite values.

The transformed representation was subsequently tested with each candidate classifier. Models that completed fitting and prediction successfully were confirmed to be compatible with the current preprocessing output.

The validated pipelines can now proceed to stratified cross-validation, where predictive performance and generalization will be evaluated.

## Cross-validation strategy

The candidate models are evaluated using stratified k-fold cross-validation on the training dataset.

A 5-fold `StratifiedKFold` strategy is used to divide the training data into five subsets while approximately preserving the original distribution of the `Churn` target in every fold.

During each iteration, four folds are used to fit the complete modeling pipeline and the remaining fold is used as validation data. The validation fold changes at every iteration until each observation has been evaluated once as unseen data.

Because preprocessing is included inside each model pipeline, the preprocessing stage is fitted only on the training portion of each fold. The corresponding validation fold is transformed using the preprocessing parameters learned from that fold's training data. This prevents preprocessing leakage during cross-validation.

The held-out test set is not used during this stage and remains reserved for the final evaluation of the selected model.

In [151]:
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [152]:
existing_classes = sorted(y_train.unique())

base_metrics = {
    "precision": precision_score,
    "recall": recall_score,
    "f1": f1_score,
}

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

for metric_name, metric_function in base_metrics.items():
    for classes in existing_classes:
        scoring[f"{metric_name}_class_##{classes}##"] = make_scorer(metric_function, pos_label=classes)

cv_results = {}

for model_name, pipeline in candidate_models_pipeline.items():
    cv_results[model_name] = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv_strategy,
        scoring=scoring,
        return_train_score=True
    )

    print(f"{model_name} evaluated successfully.")

extra_trees evaluated successfully.
gradient_boosting evaluated successfully.
hist_gradient_boosting evaluated successfully.
support_vector_machine evaluated successfully.
knn evaluated successfully.
gaussian_naive_bayes evaluated successfully.


/Users/emiliogarcialopez/projectGitHub/telco-churn-project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/emiliogarcialopez/projectGitHub/telco-churn-project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/emiliogarcialopez/projectGitHub/telco-churn-project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/emiliogarcialopez/projectGitHub/telco-churn-project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: Con

mlp evaluated successfully.
xgboost evaluated successfully.
lightgbm evaluated successfully.
catboost evaluated successfully.


In [153]:
summary = []

for model_name, results in cv_results.items():
    row = {"model": model_name}

    for key, values in results.items():
        if key.startswith("test_") or key.startswith("train_"):
            row[f"{key}_mean"] = values.mean()
            row[f"{key}_std"] = values.std()

    summary.append(row)

candidate_cv_summary = pd.DataFrame(summary).set_index("model").T
display(candidate_cv_summary.head(8))

model,extra_trees,gradient_boosting,hist_gradient_boosting,support_vector_machine,knn,gaussian_naive_bayes,mlp,xgboost,lightgbm,catboost
test_accuracy_mean,0.765710,0.803518,0.795352,0.804935,0.761804,0.696664,0.783104,0.784347,0.796950,0.798014
test_accuracy_std,0.008901,0.013616,0.009343,0.005969,0.004638,0.009061,0.005962,0.012038,0.012779,0.011680
train_accuracy_mean,0.998314,0.831337,0.895412,0.822817,0.837948,0.696530,0.864351,0.952964,0.897675,0.882632
train_accuracy_std,0.000226,0.003354,0.002496,0.002510,0.004727,0.001350,0.003972,0.004400,0.001852,0.002593
test_balanced_accuracy_mean,0.668557,0.715868,0.704969,0.707860,0.686194,0.745053,0.708381,0.698335,0.708407,0.705072
test_balanced_accuracy_std,0.014162,0.018301,0.014233,0.008611,0.007036,0.009173,0.012278,0.014018,0.016120,0.015546
train_balanced_accuracy_mean,0.996983,0.751732,0.848647,0.732795,0.779787,0.744747,0.813727,0.933590,0.852003,0.825581
train_balanced_accuracy_std,0.000436,0.005979,0.004548,0.006130,0.005797,0.002255,0.010865,0.006740,0.004977,0.004281


In [154]:
display(baseline_cv_summary.head(5))
display(candidate_cv_summary.head(5))

,dummy,logistic_regression,decision_tree,random_forest
test_accuracy_mean,0.734647,0.802451,0.727901,0.785234
test_accuracy_std,0.000094,0.011979,0.009219,0.009333
train_accuracy_mean,0.734647,0.806310,0.998314,0.998314
train_accuracy_std,0.000024,0.003327,0.000226,0.000226
test_balanced_accuracy_mean,0.500000,0.719843,0.652011,0.685906


model,extra_trees,gradient_boosting,hist_gradient_boosting,support_vector_machine,knn,gaussian_naive_bayes,mlp,xgboost,lightgbm,catboost
test_accuracy_mean,0.765710,0.803518,0.795352,0.804935,0.761804,0.696664,0.783104,0.784347,0.796950,0.798014
test_accuracy_std,0.008901,0.013616,0.009343,0.005969,0.004638,0.009061,0.005962,0.012038,0.012779,0.011680
train_accuracy_mean,0.998314,0.831337,0.895412,0.822817,0.837948,0.696530,0.864351,0.952964,0.897675,0.882632
train_accuracy_std,0.000226,0.003354,0.002496,0.002510,0.004727,0.001350,0.003972,0.004400,0.001852,0.002593
test_balanced_accuracy_mean,0.668557,0.715868,0.704969,0.707860,0.686194,0.745053,0.708381,0.698335,0.708407,0.705072


In [155]:
# Make sure metric columns are numeric
candidate_metric_columns = [
    column
    for column in candidate_cv_summary.columns
    if column != "model"
]

candidate_cv_summary[candidate_metric_columns] = (
    candidate_cv_summary[candidate_metric_columns]
    .apply(pd.to_numeric)
)

display(candidate_cv_summary.head())

model,extra_trees,gradient_boosting,hist_gradient_boosting,support_vector_machine,knn,gaussian_naive_bayes,mlp,xgboost,lightgbm,catboost
test_accuracy_mean,0.765710,0.803518,0.795352,0.804935,0.761804,0.696664,0.783104,0.784347,0.796950,0.798014
test_accuracy_std,0.008901,0.013616,0.009343,0.005969,0.004638,0.009061,0.005962,0.012038,0.012779,0.011680
train_accuracy_mean,0.998314,0.831337,0.895412,0.822817,0.837948,0.696530,0.864351,0.952964,0.897675,0.882632
train_accuracy_std,0.000226,0.003354,0.002496,0.002510,0.004727,0.001350,0.003972,0.004400,0.001852,0.002593
test_balanced_accuracy_mean,0.668557,0.715868,0.704969,0.707860,0.686194,0.745053,0.708381,0.698335,0.708407,0.705072


## Combine baseline and candidate model results

The cross-validation results of the candidate models are summarized using the mean and standard deviation of each training and validation metric across the five folds.

The mean represents the model's average performance, while the standard deviation measures its variability across different data partitions. A lower standard deviation indicates more consistent performance, whereas a larger value suggests that the model is more sensitive to the composition of the folds.

Before combining the results, the candidate summary is reorganized so that each row represents one model and each column represents one evaluation statistic. The metric columns are converted to numeric values to ensure that comparisons, sorting, and calculations can be performed correctly.

The candidate and baseline summaries are then validated to confirm that they contain the same metric columns. Their column order is aligned before concatenation, producing a single comparison table containing all evaluated models.

This combined table will be used to compare:

- Average validation performance.
- Performance variability across folds.
- Differences between training and validation scores.
- Possible overfitting or underfitting.
- Performance on the churn class (`1`).
- Overall discrimination through ROC-AUC and PR-AUC.

The principal model-selection metrics are churn recall, churn F1-score, PR-AUC, and balanced accuracy. Accuracy is included for context but will not be used alone because the target distribution is imbalanced.

In [156]:
all_cv_summary = pd.concat([baseline_cv_summary, candidate_cv_summary],axis="columns")

display(all_cv_summary)

,dummy,logistic_regression,decision_tree,random_forest,extra_trees,gradient_boosting,hist_gradient_boosting,support_vector_machine,knn,gaussian_naive_bayes,mlp,xgboost,lightgbm,catboost
test_accuracy_mean,0.734647,0.802451,0.727901,0.785234,0.765710,0.803518,0.795352,0.804935,0.761804,0.696664,0.783104,0.784347,0.796950,0.798014
test_accuracy_std,0.000094,0.011979,0.009219,0.009333,0.008901,0.013616,0.009343,0.005969,0.004638,0.009061,0.005962,0.012038,0.012779,0.011680
train_accuracy_mean,0.734647,0.806310,0.998314,0.998314,0.998314,0.831337,0.895412,0.822817,0.837948,0.696530,0.864351,0.952964,0.897675,0.882632
train_accuracy_std,0.000024,0.003327,0.000226,0.000226,0.000226,0.003354,0.002496,0.002510,0.004727,0.001350,0.003972,0.004400,0.001852,0.002593
test_balanced_accuracy_mean,0.500000,0.719843,0.652011,0.685906,0.668557,0.715868,0.704969,0.707860,0.686194,0.745053,0.708381,0.698335,0.708407,0.705072
test_balanced_accuracy_std,0.000000,0.019918,0.005583,0.015227,0.014162,0.018301,0.014233,0.008611,0.007036,0.009173,0.012278,0.014018,0.016120,0.015546
train_balanced_accuracy_mean,0.500000,0.725352,0.996983,0.997571,0.996983,0.751732,0.848647,0.732795,0.779787,0.744747,0.813727,0.933590,0.852003,0.825581
train_balanced_accuracy_std,0.000000,0.005413,0.000436,0.000271,0.000436,0.005979,0.004548,0.006130,0.005797,0.002255,0.010865,0.006740,0.004977,0.004281
test_roc_auc_mean,0.500000,0.846158,0.652240,0.818663,0.785014,0.848120,0.836436,0.797729,0.781177,0.821208,0.815597,0.824643,0.837337,0.842369
test_roc_auc_std,0.000000,0.012581,0.006211,0.012128,0.016281,0.012601,0.005448,0.017621,0.006695,0.011912,0.012355,0.006814,0.007867,0.008960


In [157]:
# Number of models
model_count = all_cv_summary.shape[1]

# Model names must be unique
assert all_cv_summary.columns.is_unique

# No missing model names
assert all_cv_summary.columns.notna().all()

print(f"Model count: {model_count}")
print("No duplicated model names found.")

model_count = all_cv_summary.shape[1]

duplicated_models = all_cv_summary.columns[
    all_cv_summary.columns.duplicated()
].tolist()

assert len(duplicated_models) == 0, (
    f"Duplicated models found: {duplicated_models}"
)

print(f"Model count: {model_count}")
print("Model validation completed successfully.")

Model count: 14
No duplicated model names found.
Model count: 14
Model validation completed successfully.


### Combined-results verification

The baseline and candidate model summaries were combined successfully. Every row represents one evaluated model, and all models share the same cross-validation metrics and summary statistics.

The combined table provides a consistent basis for comparing model performance and stability. Validation means measure expected generalization performance, validation standard deviations measure consistency across folds, and training–validation differences help identify possible overfitting.

The next analysis will rank the models according to the metrics most relevant to churn detection and identify the strongest candidates for hyperparameter tuning.

## Cross-validation performance and variability comparison

The baseline and candidate models are compared using both their **mean cross-validation performance** and their **variability across folds**.

The mean cross-validation score represents the average predictive performance obtained across the validation folds, while the standard deviation measures how much that performance changes between folds. Evaluating both values is important because a model with a strong average score may still be unreliable if its performance varies substantially across different subsets of the training data.

The comparison focuses primarily on metrics relevant to the churn-detection objective, including **recall for the churn class, F1-score, balanced accuracy, and PR-AUC**, while accuracy and ROC-AUC provide additional context.

For each metric:

- A **higher mean score** indicates better average predictive performance.
- A **lower standard deviation** indicates more stable performance across folds.
- A strong candidate should therefore combine **high validation performance with relatively low variability**.

Model selection will not be based on a single metric. The objective is to identify models that provide a favorable balance between churn detection performance, stability across folds, and generalization behavior. The strongest candidates will then be examined further before hyperparameter tuning and final evaluation on the held-out test set.

In [165]:
comparison_metrics = [
    "train_f1_class_##1##_mean",
    "test_f1_class_##1##_mean",
    "test_f1_class_##1##_std",

    "train_recall_class_##1##_mean",
    "test_recall_class_##1##_mean",
    "test_recall_class_##1##_std",

    "test_balanced_accuracy_mean",
    "test_balanced_accuracy_std",

    "train_recall_class_##0##_mean",
    "test_recall_class_##0##_mean",
    "test_recall_class_##0##_std"
]

cv_performance_comparison = all_cv_summary.loc[comparison_metrics]

model_order = (
    cv_performance_comparison
    .loc["test_f1_class_##1##_mean"]
    .sort_values(ascending=False)
    .index
)

cv_performance_comparison = cv_performance_comparison[model_order]

display(cv_performance_comparison.T.round(3))

,train_f1_class_##1##_mean,test_f1_class_##1##_mean,test_f1_class_##1##_std,train_recall_class_##1##_mean,test_recall_class_##1##_mean,test_recall_class_##1##_std,test_balanced_accuracy_mean,test_balanced_accuracy_std,train_recall_class_##0##_mean,test_recall_class_##0##_mean,test_recall_class_##0##_std
gaussian_naive_bayes,0.597,0.597,0.009,0.847,0.848,0.021,0.745,0.009,0.642,0.642,0.015
logistic_regression,0.602,0.593,0.030,0.553,0.544,0.041,0.720,0.020,0.898,0.896,0.011
gradient_boosting,0.647,0.588,0.030,0.582,0.529,0.030,0.716,0.018,0.921,0.903,0.011
support_vector_machine,0.618,0.577,0.014,0.541,0.501,0.018,0.708,0.009,0.925,0.915,0.008
lightgbm,0.796,0.576,0.026,0.755,0.520,0.025,0.708,0.016,0.949,0.897,0.010
mlp,0.734,0.573,0.018,0.706,0.549,0.030,0.708,0.012,0.922,0.868,0.010
catboost,0.761,0.571,0.025,0.704,0.507,0.027,0.705,0.016,0.947,0.903,0.011
hist_gradient_boosting,0.792,0.570,0.023,0.749,0.512,0.028,0.705,0.014,0.948,0.898,0.009
xgboost,0.910,0.559,0.022,0.892,0.515,0.024,0.698,0.014,0.975,0.882,0.014
random_forest,0.997,0.539,0.025,0.996,0.474,0.031,0.686,0.015,0.999,0.898,0.008


## Shortlist of candidate models

Based on the cross-validation comparison, the five strongest candidate models are selected for further analysis.

The shortlist is primarily determined using the mean F1-score for the churn class, while also considering cross-validation variability and the difference between training and validation performance.

The selected models are:

- Gaussian Naive Bayes
- Logistic Regression
- Gradient Boosting
- Support Vector Machine
- LightGBM

These models represent different modeling families and provide competitive cross-validation performance under the current preprocessing and evaluation framework.

The next stage will investigate whether their performance can be improved through model-specific hyperparameter tuning while monitoring generalization, variability, and the trade-off between precision and recall for the churn class.

In [168]:
top_5_models = (cv_performance_comparison.T.head(5).index.tolist())

top_5_models

['gaussian_naive_bayes',
 'logistic_regression',
 'gradient_boosting',
 'support_vector_machine',
 'lightgbm']

In [170]:
cv_performance_comparison[top_5_models]

,gaussian_naive_bayes,logistic_regression,gradient_boosting,support_vector_machine,lightgbm
train_f1_class_##1##_mean,0.597107,0.602314,0.646782,0.618266,0.796459
test_f1_class_##1##_mean,0.597406,0.593078,0.588192,0.576711,0.575979
test_f1_class_##1##_std,0.009343,0.029801,0.029804,0.013941,0.026356
train_recall_class_##1##_mean,0.847492,0.552843,0.582107,0.540970,0.754682
test_recall_class_##1##_mean,0.848161,0.543813,0.529097,0.501003,0.519732
test_recall_class_##1##_std,0.021426,0.040885,0.030462,0.018367,0.024722
test_balanced_accuracy_mean,0.745053,0.719843,0.715868,0.707860,0.708407
test_balanced_accuracy_std,0.009173,0.019918,0.018301,0.008611,0.016120
train_recall_class_##0##_mean,0.642003,0.897862,0.921358,0.924620,0.949323
test_recall_class_##0##_mean,0.641946,0.895873,0.902638,0.914717,0.897081


## Final candidate selection

The cross-validation results were evaluated using predictive performance, variability across folds, class-specific recall, and train-validation generalization gaps.

Five models achieved competitive F1 performance for the churn class and were examined in greater detail: Gaussian Naive Bayes, Logistic Regression, Gradient Boosting, Support Vector Machine, and LightGBM.

Gaussian Naive Bayes produced the highest mean F1-score and recall for the churn class while showing very low cross-validation variability and virtually no train-validation gap. However, its substantially lower recall for class 0 indicates that its strong churn detection performance is obtained by sacrificing performance on non-churn customers.

Logistic Regression showed one of the strongest overall generalization profiles. Its training and validation scores remain very close, while its class-specific recall values are considerably more balanced than those of Gaussian Naive Bayes.

Gradient Boosting also achieved competitive validation performance, although the larger train-validation gap indicates some overfitting. Because its validation performance remains competitive, further regularization and hyperparameter tuning may improve its generalization.

Support Vector Machine demonstrated relatively stable cross-validation performance and a moderate train-validation gap, but its current F1-score and recall for the churn class are slightly below the strongest candidates.

LightGBM showed the largest train-validation gap among the shortlisted models. Its substantially higher training performance indicates strong overfitting under the current default configuration. It may still be investigated through stronger regularization, but its current results do not justify selecting it as a primary finalist.

Based on the combined performance, stability, and generalization analysis, the primary finalist models are:

- Gaussian Naive Bayes
- Logistic Regression
- Gradient Boosting

Support Vector Machine and LightGBM will remain secondary candidates. SVM may be retained as a stable alternative, while LightGBM will require explicit overfitting control before it can compete with the primary finalists.

The next modeling stage will focus on model-specific hyperparameter optimization, precision-recall trade-offs, and threshold analysis while continuing to preserve the held-out test set for final evaluation.